# Malfunction Detection Algorithm

## Combined Leakage Algorithm

### Load Data

1. Import data pre-processed in Matlab

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [ ]:
df = pd.read_csv("model.csv")

print(df.shape)
print(df.columns)

In [ ]:
pd.set_option('display.max_columns', None)
cols_to_drop = [
    "Buildup_mean_slope","Buildup_mean_distance","Buildup_area_between","Buildup_correlation",
    "Buildup_std_slope","Buildup_mean_angle","Buildup_std_angle","Buildup_max_distance",
    "Buildup_max_deltap","Buildup_mean_delay","Buildup_var_distance","Buildup_normalized_area",
    "Buildup_diff_area","Holding_mean_slope","Holding_mean_distance","Holding_area_between",
    "Holding_correlation","Holding_std_slope","Holding_mean_angle","Holding_std_angle",
    "Holding_max_distance","Holding_max_deltap","Holding_mean_delay","Holding_var_distance",
    "Holding_normalized_area","Holding_diff_area","Release_mean_slope","Release_mean_distance",
    "Release_area_between","Release_correlation","Release_std_slope","Release_mean_angle",
    "Release_std_angle","Release_max_distance","Release_max_deltap","Release_mean_delay",
    "Release_var_distance","Release_normalized_area","Release_diff_area","Max_pressure_delay"
]

df = df.drop(columns=cols_to_drop, errors="ignore")

df = df.loc[:, ~df.columns.str.endswith('_model')]
df.info()
df_raw = df.copy()
df.shape

In [ ]:
df.head()

### Data cleaning and pre-processing

1. Since the goal in this phase is feature selection for malfunction detection, the sensors subjected to both healthy and combined leakages condition will be considered

In [ ]:
# Comparison with the sensors associated with leakages
level = [1,3]
df = df[df['Sensor'].isin(level)]

2. Switch to a binary classification problem: Healthy vs Combined_leakage

In [ ]:
# Keep the original Malfunction column as-is
df['Malfunction'] = df['Malfunction'].astype(str)

# Add grouped labels for binary classification
df['LeakageLabel'] = np.where(df['Malfunction'].isin(['C','D','E','F','G']),
                              'Combined leakage', 'Healthy')

In [ ]:
df_raw['Malfunction'].value_counts()

3. Features: data associated with the extracted features
   Parameters: main parameters influencing the features for malfunction detection
   Target: malfunction to detect (label problem)

In [ ]:
Features = df.drop(columns=['Malfunction','LeakageLabel','Weight','Brake_action','Brake_mode','Frequency','Sensor'])
Parameters = df[['Weight','Brake_action','Brake_mode','Frequency','Sensor']]
Target = df['LeakageLabel']
Target_raw = df['Malfunction']

4. Data filtering: 1) Replace the missing variable with the group median 2) Delete all features with constant values or all NaN

In [ ]:
# Impute missing (with median)
imp = SimpleImputer(strategy="median")
Features = pd.DataFrame(imp.fit_transform(Features), columns=Features.columns, index=Features.index)
#Features = Features.dropna(axis=1)

# Drop columns that are all NaN or constant
Features = Features.loc[:, Features.notna().any()]  # drop all-NaN
const_mask = Features.nunique(dropna=True) <= 1
if const_mask.any():
    Features = Features.loc[:, ~const_mask]

Features.shape
Features.columns

5. Feature Engineering: the features calculated for each phases are aggreated toghether to provide features features associated to the full braking action. As already described in previous paper, combined leakages have an effect on the global behaviour of the brake cylinder

In [ ]:
Features['Total_timing_delay'] = Features['Brake_timing_delay_exp'] + Features['Release_timing_delay_exp']
Features['Total_energy_delay'] = Features['Brake_energy_delay_exp'] + Features['Release_energy_delay_exp']
Features['Total_power_delay'] = Features['Brake_power_delay_exp'] + Features['Release_power_delay_exp']
Features['Total_power_efficiency'] = Features['Brake_power_efficiency_exp'] + Features['Release_power_efficiency_exp']
Features['Total_energy_efficiency'] = Features['Brake_energy_effiency_exp'] + Features['Release_energy_efficiency_exp']
Features = Features.drop(columns=['Brake_power_efficiency_exp','Release_power_efficiency_exp','Brake_energy_effiency_exp','Release_energy_efficiency_exp','Brake_timing_delay_exp','Release_timing_delay_exp','Brake_energy_delay_exp','Release_energy_delay_exp','Brake_power_delay_exp','Release_power_delay_exp'])
Features.head()

In [ ]:
Features['Total_energy_BC'] = Features['Brake_energy_exp']+Features['Release_energy_exp']
Features['Total_energy_MBP'] = Features['Brake_energy_pipe']+Features['Release_energy_pipe']
Features['Total_time_BC'] = Features['Brake_timing_exp']+Features['Release_timing_exp']
Features['Total_time_MBP'] = Features['Brake_timing_pipe']+Features['Release_timing_pipe']
Features['Pressure_ratio'] = Features['Max_pressure_exp']/ Features['Max_pressure_pipe']
# Features['Total_power_normalized'] = Features['Total_power_efficiency'] * Features['Pressure_ratio']
# Features['Energy'] = Features['Total_energy_BC']/Features['Total_energy_MBP']

Features = Features.drop(columns=['Pressure_ratio'])

In [ ]:
Features.head()

### Feature Importance and Feature Selection

1. Random Forest Classifier: https://www.datacamp.com/tutorial/random-forests-classifier-python

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

rf = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
rf.fit(Features, Target)

# Get feature importances: GIni as default, selection based on the reduction of entrophy/increase in accuracy
importances = pd.Series(rf.feature_importances_, index=Features.columns)
importances = importances.sort_values(ascending=False)
top10_features = importances.sort_values(ascending=False).head(10).index.tolist()
# Show top 5 features
print(importances.head(10))

# Plot
plt.figure(figsize=(8,6))
importances.head(10).plot(kind='barh')
plt.gca().invert_yaxis()
plt.title("Top 5 Feature Importances (RandomForest)")
plt.show()

2. Anova F-Test: https://www.datacamp.com/tutorial/anova-test

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=5)
selector.fit(Features, Target)
scores = pd.Series(selector.scores_, index=Features.columns)
print(scores.sort_values(ascending=False).head(5))

top_5_scores = scores.sort_values(ascending=False).head(5)
plt.figure(figsize=(8,6))
top_5_scores.plot(kind='barh')
plt.gca().invert_yaxis() 
plt.title("Top 5 Feature Importances (SelectKBest F-Value)") 
plt.show()

3. Combination of different feature selection methods.
Additional method: Mutual Information https://www.blog.trainindata.com/mutual-information-with-python/

In [ ]:
# === Unified Feature Selection Pipeline: RF, MI, ANOVA
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import mutual_info_classif, f_classif, RFE, RFECV
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt

Xsel = Features.copy()

# Keep only numeric columns (if any non-numeric slipped in)
num_cols = [c for c in Xsel.columns if np.issubdtype(Xsel[c].dtype, np.number)]
Xsel = Xsel[num_cols].copy()

# Some selectors need scaling
sc_std = StandardScaler()
sc_rob = RobustScaler()
X_std = pd.DataFrame(sc_std.fit_transform(Xsel), columns=Xsel.columns, index=Xsel.index)
X_rob = pd.DataFrame(sc_rob.fit_transform(Xsel), columns=Xsel.columns, index=Xsel.index)

# if y is string, change to 0/1
y_enc = pd.Series(Target).astype("category")
if y_enc.dtype.name == "category":
    y_enc = y_enc.cat.codes  # e.g., Leakage=1, Normal=0

# For stability on tiny datasets
cv = StratifiedKFold(n_splits=min(5, max(2, np.bincount(y_enc).min())), shuffle=True, random_state=42)

# Helper to convert scores to ranks (lower rank = better)
def to_rank(series, higher_is_better=True):
    s = series.copy()
    if not higher_is_better:
        s = -s
    # rank 1 = best
    return s.rank(ascending=False, method="average")

# ---------- 1) RandomForest importance ----------
# Measures a feature's utility in improving the model's prediction accuracy (e.g., mean decrease in impurity).
# Captures feature interactions naturally; highly effective.
rf = RandomForestClassifier(n_estimators=500, random_state=42, class_weight="balanced")
rf.fit(Xsel, y_enc)
rf_imp = pd.Series(rf.feature_importances_, index=Xsel.columns, name="RF_Importance")
rf_rank = to_rank(rf_imp, higher_is_better=True).rename("RF_Rank")

# ---------- 2) Mutual Information ----------
# Measures statistical dependency (information gain) between a feature and the target.
# Captures non-linear relationships. Evaluates each feature independently; ignores feature interactions.
mi = mutual_info_classif(Xsel, y_enc, random_state=42, discrete_features=False)
mi_score = pd.Series(mi, index=Xsel.columns, name="MI_Score")
mi_rank = to_rank(mi_score, higher_is_better=True).rename("MI_Rank")

# ---------- 3) ANOVA F-test ----------
# (works best if roughly Gaussian/scaled; we used imputed data)
# Measures linear correlation between a feature and the target by comparing variance between groups to variance within groups.
# Assumes linear relationship and Gaussian distribution; ignores feature interactions.
F_vals, p_vals = f_classif(X_std, y_enc)
f_score = pd.Series(F_vals, index=X_std.columns, name="ANOVA_F")
f_rank = to_rank(f_score, higher_is_better=True).rename("ANOVA_Rank")

# ---------- Combine all rankings ----------
rank_table = pd.concat([rf_rank, mi_rank, f_rank,
                        rf_imp, mi_score, f_score], axis=1)

# OverallRank: average of available ranks (lower = better)
rank_cols = ["RF_Rank","MI_Rank","ANOVA_Rank"]
rank_table["OverallRank"] = rank_table[rank_cols].mean(axis=1)

# Sort and display top-N
N = 10
rank_table_sorted = rank_table.sort_values("OverallRank").head(N)
print("=== Top features by OverallRank (lower = better) ===")
display(rank_table_sorted)

plt.figure(figsize=(8, max(4, 0.35*N)))
rank_table_sorted.sort_values("OverallRank")["OverallRank"].plot(kind="barh")
plt.gca().invert_yaxis()
plt.title(f"Top {N} Features by Rank")
plt.xlabel("Rank (lower is better)")
plt.tight_layout()
plt.show()

topN_features = rank_table_sorted.index.tolist()
print("\nTopN feature list:", topN_features)

Features_reduced = Features[topN_features]

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.preprocessing import StandardScaler

# --- assume Features (DataFrame) and Target exist; and NaNs already imputed upstream ---

Xsel = Features.copy()
num_cols = [c for c in Xsel.columns if np.issubdtype(Xsel[c].dtype, np.number)]
Xsel = Xsel[num_cols].copy()

# encode y
y_enc = pd.Series(Target).astype("category").cat.codes.values

# CV with safe n_splits for small datasets
n_splits = min(5, max(2, np.bincount(y_enc).min()))
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

def to_rank(series):
    # rank 1 = best (highest score)
    return series.rank(ascending=False, method="average")

# store per-fold ranks
rf_ranks = []
mi_ranks = []
an_ranks = []

for train_idx, _ in cv.split(Xsel, y_enc):
    X_tr = Xsel.iloc[train_idx]
    y_tr = y_enc[train_idx]

    # -------- RF importance (no scaling) --------
    rf = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        class_weight="balanced"
    )
    rf.fit(X_tr, y_tr)
    rf_imp = pd.Series(rf.feature_importances_, index=Xsel.columns)
    rf_ranks.append(to_rank(rf_imp))

    # -------- MI (use raw features typically) --------
    mi = mutual_info_classif(X_tr, y_tr, random_state=42, discrete_features=False)
    mi_score = pd.Series(mi, index=Xsel.columns)
    mi_ranks.append(to_rank(mi_score))

    # -------- ANOVA (prefer standardized features) --------
    sc = StandardScaler()
    X_tr_std = sc.fit_transform(X_tr)
    F_vals, _ = f_classif(X_tr_std, y_tr)
    an_score = pd.Series(F_vals, index=Xsel.columns)
    an_ranks.append(to_rank(an_score))

# aggregate ranks across folds (mean rank = stability-aware)
rf_rank_cv = pd.concat(rf_ranks, axis=1).mean(axis=1).rename("RF_Rank_CV")
mi_rank_cv = pd.concat(mi_ranks, axis=1).mean(axis=1).rename("MI_Rank_CV")
an_rank_cv = pd.concat(an_ranks, axis=1).mean(axis=1).rename("ANOVA_Rank_CV")

rank_table_cv = pd.concat([rf_rank_cv, mi_rank_cv, an_rank_cv], axis=1)

# ---- weighted overall rank (give more weight to ANOVA) ----
w_rf, w_mi, w_an = 0.2, 0.2, 0.6   # example: ANOVA emphasized
rank_table_cv["OverallRank_CV"] = (
    w_rf * rank_table_cv["RF_Rank_CV"] +
    w_mi * rank_table_cv["MI_Rank_CV"] +
    w_an * rank_table_cv["ANOVA_Rank_CV"]
)

# top-N
N = 10
rank_table_sorted = rank_table_cv.sort_values("OverallRank_CV").head(N)
print("=== Top features by OverallRank (lower = better) ===")
display(rank_table_sorted)

plt.figure(figsize=(8, max(4, 0.35*N)))
rank_table_sorted.sort_values("OverallRank_CV")["OverallRank_CV"].plot(kind="barh")
plt.gca().invert_yaxis()
plt.title(f"Top {N} Features by Rank")
plt.xlabel("Rank (lower is better)")
plt.tight_layout()
plt.show()

topN_features = rank_table_sorted.index.tolist()
print("\nTopN feature list:", topN_features)

Features_reduced = Features[topN_features]


4. Features correlations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.heatmap(Features_reduced.corr(), annot=True, cmap='coolwarm')
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
selected_features = ["Total_power_efficiency","Total_power_delay","Total_energy_delay","Std_delay_exp","Total_energy_efficiency"]
Features_reduced = Features[selected_features]
sns.heatmap(Features_reduced.corr(), annot=True, cmap='coolwarm')
plt.title("Feature Correlation Heatmap")
plt.show()

5. Parameters correlation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.heatmap(Parameters.corr(), annot=True, cmap='coolwarm')
plt.title("Parameters Correlation Heatmap")
plt.show()

### Features vs target relations

In [ ]:
def box_target_plotter(data, target):
    for col in data.select_dtypes("number"):
        sns.boxplot(data=data, x=target, y=col)
        plt.show()

box_target_plotter(Features_reduced, Target)

In [ ]:
def box_target_plotter(data, target, target_order=None):
    for col in data.select_dtypes("number"):
        sns.boxplot(data=data, x=target, y=col, order=target_order)
        plt.show()

desired_order = ['0', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
box_target_plotter(Features_reduced, Target_raw, target_order=desired_order)

In [ ]:
group_map = {
    '0': 'Healthy',
    'H': 'Healthy',
    'A': 'Auxiliary Leakage',
    'B': 'Auxiliary Leakage',
    'C': 'WV Leakage',
    'D': 'WV Leakage',
    'E': 'WV Leakage',
    'F': 'WV Leakage',
    'G': 'WV Leakage'
}
Target_grouped = Target_raw.map(group_map)

In [ ]:
def box_target_plotter(data, target, target_order=None):
    for col in data.select_dtypes("number"):
        sns.boxplot(data=data, x=target, y=col, order=target_order)
        plt.show()

desired_order_grouped = ['Healthy', 'Auxiliary Leakage', 'WV Leakage']
box_target_plotter(Features_reduced, Target_grouped, target_order=desired_order_grouped)

### Parameters influence

1. Brake mode influence on each features for malfunction detection

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import f_classif
from itertools import product
import numpy as np

levels_weight = Parameters['Weight'].unique()
levels_brake_action = Parameters['Brake_action'].unique()
combinations = list(product(levels_weight, levels_brake_action))

Study_parameter = 'Brake_mode' 
Target_name = Target.name if Target.name else 'Target_Y'
Custom_hue = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

results_column = [
    'Weight', 'Action', 'Top_Feature', 'Max_F_Score',
    'Separability_no_mode',
    'Max_separability_mode_freight', 'Max_separability_mode_passenger', 'Min_separability',
    'Influence_mode_freight', 'Influence_mode_passenger' 
]
df_results = pd.DataFrame(columns=results_column)

for Weight, Action in combinations:
    
    condition_mask = (Parameters['Brake_action'] == Action) & \
                     (Parameters['Weight'] == Weight)
    
    Parameters_controlled = Parameters.loc[condition_mask].copy()
    Features_controlled = Features_reduced.loc[condition_mask].copy()
    Target_controlled = Target.loc[condition_mask].copy()

    if Parameters_controlled.empty:
        continue

    Features_controlled[Study_parameter] = Parameters_controlled[Study_parameter]
    Features_controlled[Target_name] = Target_controlled.values 
    feature_columns = Features_controlled.columns.drop([Study_parameter, Target_name])

    Y_analisi = Features_controlled[Target_name]
    X_analisi = Features_controlled[feature_columns]
    
    if len(Y_analisi.unique()) < 2:
        continue 
    
    # F-score for top feature
    F_vals, p_vals = f_classif(X_analisi, Y_analisi)
    scores = pd.Series(F_vals, index=X_analisi.columns)
    top_scores = scores.sort_values(ascending=False)
    
    max_f_score = top_scores.iloc[0]
    top_feature_name = top_scores.index[0]
    
    # Influence indicator
    mediane_target_senza = Features_controlled.groupby(Target_name)[top_feature_name].median()
    target_levels = mediane_target_senza.index.tolist()
    
    if len(target_levels) != 2: continue # If the Target are not 2 classes
        
    Target_1 = target_levels[0]
    Target_2 = target_levels[1]
    
    # Separability without brake mode
    sep_senza_mode = abs(mediane_target_senza.loc[Target_2] - mediane_target_senza.loc[Target_1])
    
    # Separability with brake mode
    mediane_nidificate = Features_controlled.groupby([Target_name, Study_parameter])[top_feature_name].median()

    try:
        mediana_T1_M0 = mediane_nidificate.loc[(Target_1, 0)]
        mediana_T2_M0 = mediane_nidificate.loc[(Target_2, 0)]
        sep_con_mode_0 = abs(mediana_T2_M0 - mediana_T1_M0)
    except KeyError:
        mediana_T1_M0 = np.nan
        mediana_T2_M0 = np.nan
        sep_con_mode_0 = np.nan
    
    try:
        mediana_T1_M1 = mediane_nidificate.loc[(Target_1, 1)]
        mediana_T2_M1 = mediane_nidificate.loc[(Target_2, 1)]
        sep_con_mode_1 = abs(mediana_T2_M1 - mediana_T1_M1)
    except KeyError:
        mediana_T1_M1 = np.nan
        mediana_T2_M1 = np.nan 
        sep_con_mode_1 = np.nan
    
    if not np.isnan(mediana_T1_M0) and not np.isnan(mediana_T2_M1) and mediana_T1_M0 != 0 and mediana_T2_M1 != 0:
        mediana_diff_T1 = abs(mediana_T1_M0 - mediana_T2_M1)
    else:
        mediana_diff_T1 = np.nan
        
    if not np.isnan(mediana_T1_M1) and not np.isnan(mediana_T2_M0) and mediana_T1_M1 != 0 and mediana_T2_M0 != 0:
        mediana_diff_T2 = abs(mediana_T1_M1 - mediana_T2_M0)
    else:
        mediana_diff_T2 = np.nan
        
    if not np.isnan(mediana_diff_T1) and not np.isnan(mediana_diff_T2):
        min_sep_con_mode = min(mediana_diff_T1, mediana_diff_T2)
    else:
        min_sep_con_mode = np.nan
    
    influenza_mode_freight = ((sep_con_mode_0 - sep_senza_mode)/sep_senza_mode)*100
    influenza_mode_passenger = ((sep_con_mode_1 - sep_senza_mode)/sep_senza_mode)*100

    nuova_riga = pd.DataFrame([{
        'Weight': Weight, 'Action': Action,
        'Top_Feature': top_feature_name,
        'Max_F_Score': max_f_score,
        'Separability_no_mode': sep_senza_mode,
        'Max_separability_mode_freight': sep_con_mode_0,
        'Max_separability_mode_passenger': sep_con_mode_1,
        'Min_separability': min_sep_con_mode,
        'Influence_mode_freight': influenza_mode_freight,
        'Influence_mode_passenger': influenza_mode_passenger
    }])
    df_results = pd.concat([df_results, nuova_riga], ignore_index=True)
    
    plt.figure(figsize=(20, 6)) 
    plt.suptitle(f"Sensitivity analysis | W={Weight}, A={Action} | Feature: {top_feature_name}", fontsize=16)

    # F-Scores
    plt.subplot(1, 3, 1) 
    top_scores.head(5).plot(kind='barh', color='#1f77b4')
    plt.gca().invert_yaxis()
    plt.title("1. Top 5 F-Scores")
    plt.xlabel("F-Score Value")

    # Top Feature vs. Target no Mode
    plt.subplot(1, 3, 2)
    sns.boxplot(data=Features_controlled, x=Target_name, y=top_feature_name, color='#ff7f0e')
    plt.title(f"2. {top_feature_name} vs. Target (Sep. {sep_senza_mode:.3f})")
    plt.xlabel(f"Target ({Target_name})")
    plt.ylabel(top_feature_name)
    plt.grid(axis='y', linestyle='--')

    # Top Feature vs. Target with Mode
    plt.subplot(1, 3, 3)
    sns.boxplot(
        data=Features_controlled,
        x=Target_name,
        y=top_feature_name,
        hue=Study_parameter, 
        palette=Custom_hue
    )
    plt.title(f"3. Interation: Target x {Study_parameter}")
    plt.xlabel(f"Target ({Target_name})")
    plt.ylabel(top_feature_name)
    plt.grid(axis='y', linestyle='--')
    plt.legend(title=Study_parameter)
    
    plt.tight_layout(rect=[0, 0.05, 1, 0.9])
    plt.show()

print("\n\n=======================================================")
print("  Results: Brake mode influence on separability ")
print("=======================================================")
print(df_results)

2. Brake action influence on each features for malfunction detection

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import f_classif
from itertools import product
import numpy as np

levels_weight = Parameters['Weight'].unique()
levels_brake_action = Parameters['Brake_mode'].unique()
combinations = list(product(levels_weight, levels_brake_action))

Study_parameter = 'Brake_action' 
Target_name = Target.name if Target.name else 'Target_Y'
Custom_hue = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

results_column = [
    'Weight', 'Mode', 'Top_Feature', 'Max_F_Score',
    'Separability_no_mode',
    'Max_separability_action_service', 'Max_separability_action_emergency', 'Min_separability',
    'Influence_action_service', 'Influence_action_emergency' 
]
df_results = pd.DataFrame(columns=results_column)

for Weight, Action in combinations:
    
    condition_mask = (Parameters['Brake_mode'] == Action) & \
                     (Parameters['Weight'] == Weight)
    
    Parameters_controlled = Parameters.loc[condition_mask].copy()
    Features_controlled = Features_reduced.loc[condition_mask].copy()
    Target_controlled = Target.loc[condition_mask].copy()

    if Parameters_controlled.empty:
        continue

    Features_controlled[Study_parameter] = Parameters_controlled[Study_parameter]
    Features_controlled[Target_name] = Target_controlled.values 
    feature_columns = Features_controlled.columns.drop([Study_parameter, Target_name])

    Y_analisi = Features_controlled[Target_name]
    X_analisi = Features_controlled[feature_columns]
    
    if len(Y_analisi.unique()) < 2:
        continue 
    
    # F-score for top feature
    F_vals, p_vals = f_classif(X_analisi, Y_analisi)
    scores = pd.Series(F_vals, index=X_analisi.columns)
    top_scores = scores.sort_values(ascending=False)
    
    max_f_score = top_scores.iloc[0]
    top_feature_name = top_scores.index[0]
    
    # Influence indicator
    mediane_target_senza = Features_controlled.groupby(Target_name)[top_feature_name].median()
    target_levels = mediane_target_senza.index.tolist()
    
    if len(target_levels) != 2: continue # If the Target are not 2 classes
        
    Target_1 = target_levels[0]
    Target_2 = target_levels[1]
    
    # Separability without brake mode
    sep_senza_mode = abs(mediane_target_senza.loc[Target_2] - mediane_target_senza.loc[Target_1])
    
    # Separability with brake mode
    mediane_nidificate = Features_controlled.groupby([Target_name, Study_parameter])[top_feature_name].median()

    try:
        mediana_T1_M0 = mediane_nidificate.loc[(Target_1, 0)]
        mediana_T2_M0 = mediane_nidificate.loc[(Target_2, 0)]
        sep_con_mode_0 = abs(mediana_T2_M0 - mediana_T1_M0)
    except KeyError:
        mediana_T1_M0 = np.nan
        mediana_T2_M0 = np.nan
        sep_con_mode_0 = np.nan
    
    try:
        mediana_T1_M1 = mediane_nidificate.loc[(Target_1, 1)]
        mediana_T2_M1 = mediane_nidificate.loc[(Target_2, 1)]
        sep_con_mode_1 = abs(mediana_T2_M1 - mediana_T1_M1)
    except KeyError:
        mediana_T1_M1 = np.nan
        mediana_T2_M1 = np.nan 
        sep_con_mode_1 = np.nan
    
    if not np.isnan(mediana_T1_M0) and not np.isnan(mediana_T2_M1) and mediana_T1_M0 != 0 and mediana_T2_M1 != 0:
        mediana_diff_T1 = abs(mediana_T1_M0 - mediana_T2_M1)
    else:
        mediana_diff_T1 = np.nan
        
    if not np.isnan(mediana_T1_M1) and not np.isnan(mediana_T2_M0) and mediana_T1_M1 != 0 and mediana_T2_M0 != 0:
        mediana_diff_T2 = abs(mediana_T1_M1 - mediana_T2_M0)
    else:
        mediana_diff_T2 = np.nan
        
    if not np.isnan(mediana_diff_T1) and not np.isnan(mediana_diff_T2):
        min_sep_con_mode = min(mediana_diff_T1, mediana_diff_T2)
    else:
        min_sep_con_mode = np.nan
    
    influenza_mode_freight = ((sep_con_mode_0 - sep_senza_mode)/sep_senza_mode)*100
    influenza_mode_passenger = ((sep_con_mode_1 - sep_senza_mode)/sep_senza_mode)*100

    nuova_riga = pd.DataFrame([{
        'Weight': Weight, 'Mode': Action,
        'Top_Feature': top_feature_name,
        'Max_F_Score': max_f_score,
        'Separability_no_mode': sep_senza_mode,
        'Max_separability_action_service': sep_con_mode_0,
        'Max_separability_action_emergency': sep_con_mode_1,
        'Min_separability': min_sep_con_mode,
        'Influence_action_service': influenza_mode_freight,
        'Influence_action_emergency': influenza_mode_passenger
    }])
    df_results = pd.concat([df_results, nuova_riga], ignore_index=True)
    
    plt.figure(figsize=(20, 6)) 
    plt.suptitle(f"Sensitivity analysis | W={Weight}, M={Action} | Feature: {top_feature_name}", fontsize=16)

    # F-Scores
    plt.subplot(1, 3, 1) 
    top_scores.head(5).plot(kind='barh', color='#1f77b4')
    plt.gca().invert_yaxis()
    plt.title("1. Top 5 F-Scores")
    plt.xlabel("F-Score Value")

    # Top Feature vs. Target no Mode
    plt.subplot(1, 3, 2)
    sns.boxplot(data=Features_controlled, x=Target_name, y=top_feature_name, color='#ff7f0e')
    plt.title(f"2. {top_feature_name} vs. Target (Sep. {sep_senza_mode:.3f})")
    plt.xlabel(f"Target ({Target_name})")
    plt.ylabel(top_feature_name)
    plt.grid(axis='y', linestyle='--')

    # Top Feature vs. Target with Mode
    plt.subplot(1, 3, 3)
    sns.boxplot(
        data=Features_controlled,
        x=Target_name,
        y=top_feature_name,
        hue=Study_parameter, 
        palette=Custom_hue
    )
    plt.title(f"3. Interation: Target x {Study_parameter}")
    plt.xlabel(f"Target ({Target_name})")
    plt.ylabel(top_feature_name)
    plt.grid(axis='y', linestyle='--')
    plt.legend(title=Study_parameter)
    
    plt.tight_layout(rect=[0, 0.05, 1, 0.9])
    plt.show()

print("\n\n=======================================================")
print("  Results: Brake action influence on separability ")
print("=======================================================")
print(df_results)

3. Weight influence on each features for malfunction detection (the same is expected for frequency and sensor)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import f_classif
from itertools import product
import numpy as np

levels_weight = Parameters['Brake_action'].unique()
levels_brake_action = Parameters['Brake_mode'].unique()
combinations = list(product(levels_weight, levels_brake_action))

Study_parameter = 'Weight' 
Target_name = Target.name if Target.name else 'Target_Y'
Custom_hue = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

results_column = [
    'Action', 'Mode', 'Top_Feature', 'Max_F_Score',
    'Separability_no_mode',
    'Max_separability_weight_1', 'Max_separability_weight_2', 'Min_separability',
    'Influence_weight_1', 'Influence_weight_2' 
]
df_results = pd.DataFrame(columns=results_column)

for Weight, Action in combinations:
    
    condition_mask = (Parameters['Brake_mode'] == Action) & \
                     (Parameters['Brake_action'] == Weight)
    
    Parameters_controlled = Parameters.loc[condition_mask].copy()
    Features_controlled = Features_reduced.loc[condition_mask].copy()
    Target_controlled = Target.loc[condition_mask].copy()

    if Parameters_controlled.empty:
        continue

    Features_controlled[Study_parameter] = Parameters_controlled[Study_parameter]
    Features_controlled[Target_name] = Target_controlled.values 
    feature_columns = Features_controlled.columns.drop([Study_parameter, Target_name])

    Y_analisi = Features_controlled[Target_name]
    X_analisi = Features_controlled[feature_columns]
    
    if len(Y_analisi.unique()) < 2:
        continue 
    
    # F-score for top feature
    F_vals, p_vals = f_classif(X_analisi, Y_analisi)
    scores = pd.Series(F_vals, index=X_analisi.columns)
    top_scores = scores.sort_values(ascending=False)
    
    max_f_score = top_scores.iloc[0]
    top_feature_name = top_scores.index[0]
    
    # Influence indicator
    mediane_target_senza = Features_controlled.groupby(Target_name)[top_feature_name].median()
    target_levels = mediane_target_senza.index.tolist()
    
    if len(target_levels) != 2: continue # If the Target are not 2 classes
        
    Target_1 = target_levels[0]
    Target_2 = target_levels[1]
    
    # Separability without brake mode
    sep_senza_mode = abs(mediane_target_senza.loc[Target_2] - mediane_target_senza.loc[Target_1])
    
    # Separability with brake mode
    mediane_nidificate = Features_controlled.groupby([Target_name, Study_parameter])[top_feature_name].median()

    try:
        mediana_T1_M0 = mediane_nidificate.loc[(Target_1, 2.33)]
        mediana_T2_M0 = mediane_nidificate.loc[(Target_2, 2.33)]
        sep_con_mode_0 = abs(mediana_T2_M0 - mediana_T1_M0)
    except KeyError:
        mediana_T1_M0 = np.nan
        mediana_T2_M0 = np.nan
        sep_con_mode_0 = np.nan
    
    try:
        mediana_T1_M1 = mediane_nidificate.loc[(Target_1, 2.5686)]
        mediana_T2_M1 = mediane_nidificate.loc[(Target_2, 2.5686)]
        sep_con_mode_1 = abs(mediana_T2_M1 - mediana_T1_M1)
    except KeyError:
        mediana_T1_M1 = np.nan
        mediana_T2_M1 = np.nan 
        sep_con_mode_1 = np.nan
    
    if not np.isnan(mediana_T1_M0) and not np.isnan(mediana_T2_M1) and mediana_T1_M0 != 0 and mediana_T2_M1 != 0:
        mediana_diff_T1 = abs(mediana_T1_M0 - mediana_T2_M1)
    else:
        mediana_diff_T1 = np.nan
        
    if not np.isnan(mediana_T1_M1) and not np.isnan(mediana_T2_M0) and mediana_T1_M1 != 0 and mediana_T2_M0 != 0:
        mediana_diff_T2 = abs(mediana_T1_M1 - mediana_T2_M0)
    else:
        mediana_diff_T2 = np.nan
        
    if not np.isnan(mediana_diff_T1) and not np.isnan(mediana_diff_T2):
        min_sep_con_mode = min(mediana_diff_T1, mediana_diff_T2)
    else:
        min_sep_con_mode = np.nan
    
    influenza_mode_freight = ((sep_con_mode_0 - sep_senza_mode)/sep_senza_mode)*100
    influenza_mode_passenger = ((sep_con_mode_1 - sep_senza_mode)/sep_senza_mode)*100

    nuova_riga = pd.DataFrame([{
        'Action': Weight, 'Mode': Action,
        'Top_Feature': top_feature_name,
        'Max_F_Score': max_f_score,
        'Separability_no_mode': sep_senza_mode,
        'Max_separability_weight_1': sep_con_mode_0,
        'Max_separability_weight_2': sep_con_mode_1,
        'Min_separability': min_sep_con_mode,
        'Influence_weight_1': influenza_mode_freight,
        'Influence_weight_2': influenza_mode_passenger
    }])
    df_results = pd.concat([df_results, nuova_riga], ignore_index=True)
    
    plt.figure(figsize=(20, 6)) 
    plt.suptitle(f"Sensitivity analysis | A={Weight}, M={Action} | Feature: {top_feature_name}", fontsize=16)

    # F-Scores
    plt.subplot(1, 3, 1) 
    top_scores.head(5).plot(kind='barh', color='#1f77b4')
    plt.gca().invert_yaxis()
    plt.title("1. Top 5 F-Scores")
    plt.xlabel("F-Score Value")

    # Top Feature vs. Target no Mode
    plt.subplot(1, 3, 2)
    sns.boxplot(data=Features_controlled, x=Target_name, y=top_feature_name, color='#ff7f0e')
    plt.title(f"2. {top_feature_name} vs. Target (Sep. {sep_senza_mode:.3f})")
    plt.xlabel(f"Target ({Target_name})")
    plt.ylabel(top_feature_name)
    plt.grid(axis='y', linestyle='--')

    # Top Feature vs. Target with Mode
    plt.subplot(1, 3, 3)
    sns.boxplot(
        data=Features_controlled,
        x=Target_name,
        y=top_feature_name,
        hue=Study_parameter, 
        palette=Custom_hue
    )
    plt.title(f"3. Interation: Target x {Study_parameter}")
    plt.xlabel(f"Target ({Target_name})")
    plt.ylabel(top_feature_name)
    plt.grid(axis='y', linestyle='--')
    plt.legend(title=Study_parameter)
    
    plt.tight_layout(rect=[0, 0.05, 1, 0.9])
    plt.show()

print("\n\n=======================================================")
print("  Results: Weight influence on separability ")
print("=======================================================")
print(df_results)

## Model

In [ ]:
# === Basic setup ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, RocCurveDisplay,
    PrecisionRecallDisplay, brier_score_loss
)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.calibration import calibration_curve

# Optional for nicer plots
plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:
# Merge WV_MeanPressure from Parameters into Features
Features = Features.copy()  # optional, to avoid modifying original
Features['Weight'] = Parameters['Weight']

# Select features for training
FEATURES = ['Total_power_efficiency', 'Weight', 'Total_power_delay']
X = Features[FEATURES].copy()

# Target labels (ensure alignment)
y = Target  # make sure Target aligns row-wise with Features

# Map textual labels -> binary (0 healthy, 1 leakage)
LABEL_MAP = {
    'Healthy': 0,
    'healthy': 0,
    'Combined leakage': 1,
    'combined leakage': 1,
    0: 0, 1: 1  # in case they are already numeric
}
y_enc = pd.Series(y).map(LABEL_MAP)

# Safety check
print("Unique raw labels:", pd.Series(y).unique())
print("Encoded label counts:\n", y_enc.value_counts(dropna=False))

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.1, random_state=42, stratify=y
)

# Scale features (important for logistic regression)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

y.head()


In [ ]:
# ensure numeric
Xn_train = X_train[FEATURES].apply(pd.to_numeric, errors='coerce')

# boolean masks as numpy (avoids index alignment surprises)
m0 = (pd.Series(y_train).to_numpy() == 0)
m1 = (pd.Series(y_train).to_numpy() == 1)

n0, n1 = m0.sum(), m1.sum()
if n0 == 0 or n1 == 0:
    print(f"⚠️ One class missing in train split: healthy={n0}, leakage={n1} (try different random_state or adjust test_size).")

# === Per-class histograms (shared bins, skip empty) ===
fig, axes = plt.subplots(1, len(FEATURES), figsize=(5*len(FEATURES), 3.6))
if len(FEATURES) == 1: axes = [axes]
for ax, col in zip(axes, FEATURES):
    x0 = Xn_train.loc[m0, col].dropna().values if n0 else np.array([])
    x1 = Xn_train.loc[m1, col].dropna().values if n1 else np.array([])

    if len(x0) == 0 and len(x1) == 0:
        ax.set_title(col); ax.text(0.5, 0.5, "no data", ha='center'); continue

    allx = np.concatenate([x0, x1]) if len(x0) and len(x1) else (x0 if len(x0) else x1)
    bins = np.histogram_bin_edges(allx, bins=30)
    if len(x0): ax.hist(x0, bins=bins, alpha=0.6, label='Healthy')
    if len(x1): ax.hist(x1, bins=bins, alpha=0.6, label='Leakage')
    ax.set_title(col); ax.set_xlabel(col); ax.set_ylabel('Count')
axes[0].legend(loc='upper right'); fig.suptitle('Per-class Feature Histograms (Train)', y=1.02)
plt.tight_layout()

# === Pairwise scatter (robust) ===
cols = FEATURES
pairs = [(i, j) for i in range(len(cols)) for j in range(i+1, len(cols))]
fig, axes = plt.subplots(1, len(pairs), figsize=(5*len(pairs), 3.6)) if pairs else (plt.figure(), [])
if len(pairs) == 1: axes = [axes]

for ax, (i, j) in zip(axes, pairs):
    ci, cj = cols[i], cols[j]
    if n0: ax.scatter(Xn_train.loc[m0, ci], Xn_train.loc[m0, cj], s=12, alpha=0.7, label='Healthy')
    if n1: ax.scatter(Xn_train.loc[m1, ci], Xn_train.loc[m1, cj], s=12, alpha=0.7, label='Leakage')
    ax.set_xlabel(ci); ax.set_ylabel(cj)
if pairs: axes[0].legend()
plt.suptitle('Pairwise Scatter (Train)', y=1.02); plt.tight_layout()

# === Correlation heatmap (numeric only) ===
corr = Xn_train.corr()
fig, ax = plt.subplots(figsize=(3.8, 3.2))
im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=45, ha='right')
ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Feature Correlation'); plt.tight_layout()


Define models (with proper scaling where needed)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Use your selected features and the encoded labels (0/1) from previous cells:
# X_train, X_test, y_train, y_test already defined

models = {
    "LogReg": make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight='balanced', max_iter=2000, random_state=0)
    ),
    "SVM-RBF": make_pipeline(
        StandardScaler(),
        SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=0)
    ),
    "KNN-5": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=5)
    ),
    "GaussianNB": make_pipeline(
        StandardScaler(),  # optional; helps if features are on very different scales
        GaussianNB()
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=400, max_depth=None, class_weight='balanced_subsample', random_state=0
    ),
    "GBDT": GradientBoostingClassifier(random_state=0)
}
list(models.keys())


Cross-validated model ranking (ROC-AUC, PR-AUC, F1, BalAcc)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import average_precision_score, f1_score, balanced_accuracy_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cv_pr_auc(model, X, y, cv):
    """
    PR-AUC via cross_val_predict (out-of-fold probabilities).
    """
    if hasattr(model, "predict_proba"):
        method = "predict_proba"
    else:
        # fall back to decision_function if no proba (rare here)
        method = "decision_function"
    preds = cross_val_predict(model, X, y, cv=cv, method=method)
    if preds.ndim == 2:  # predict_proba -> use positive class column
        preds = preds[:, 1]
    return average_precision_score(y, preds)

def cv_f1_balacc(model, X, y, cv):
    """
    F1 and balanced accuracy from out-of-fold class predictions.
    """
    yhat = cross_val_predict(model, X, y, cv=cv, method="predict")
    return f1_score(y, yhat), balanced_accuracy_score(y, yhat)

rows = []
for name, mdl in models.items():
    roc_auc = cross_val_score(mdl, X_train, y_train, cv=cv, scoring="roc_auc").mean()
    pr_auc  = cv_pr_auc(mdl, X_train, y_train, cv=cv)
    f1, bal = cv_f1_balacc(mdl, X_train, y_train, cv=cv)
    rows.append([name, roc_auc, pr_auc, f1, bal])

cv_table = pd.DataFrame(rows, columns=["Model", "ROC_AUC (cv5)", "PR_AUC (cv5)", "F1 (cv5)", "BalAcc (cv5)"])
cv_table = cv_table.sort_values(by="ROC_AUC (cv5)", ascending=False).reset_index(drop=True)
cv_table


Fit all models on train, plot ROC & PR on test

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve

# Fit & collect test-set curves
roc_curves = {}
pr_curves  = {}
for name, mdl in models.items():
    mdl.fit(X_train, y_train)
    # probabilities if available; else decision scores
    if hasattr(mdl, "predict_proba"):
        s_test = mdl.predict_proba(X_test)[:, 1]
    else:
        s_test = mdl.decision_function(X_test)
    fpr, tpr, _ = roc_curve(y_test, s_test)
    prec, rec, _ = precision_recall_curve(y_test, s_test)
    roc_curves[name] = (fpr, tpr, auc(fpr, tpr))
    pr_curves[name]  = (rec, prec)  # average_precision can be added if desired

# ROC overlay
plt.figure(figsize=(5.2, 4.2))
for name, (fpr, tpr, aucv) in roc_curves.items():
    plt.plot(fpr, tpr, label=f"{name} (AUC={aucv:.3f})")
plt.plot([0,1], [0,1], 'k--', lw=1)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("Test ROC Curves")
plt.legend(loc="lower right")
plt.tight_layout()

# PR overlay
plt.figure(figsize=(5.2, 4.2))
for name, (rec, prec) in pr_curves.items():
    plt.plot(rec, prec, label=name)
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title("Test Precision–Recall Curves")
plt.legend(loc="best")
plt.tight_layout()


## Manual Brake Activation

In [ ]:
# Comparison with the sensors associated with leakages
level = [2]
df = df_raw[df_raw['Sensor'].isin(level)]

In [ ]:
# Keep the original Malfunction column as-is
df['Malfunction'] = df['Malfunction'].astype(str)

# Add grouped labels for binary classification
df['ManualLabel'] = np.where(df['Malfunction'].isin(['H']),
                              'Manual Brake', 'Healthy')

In [ ]:
Features = df.drop(columns=['Malfunction','ManualLabel','Weight','Brake_action','Brake_mode','Frequency','Sensor'])
Parameters = df[['Weight','Brake_action','Brake_mode','Frequency','Sensor']]
Target = df['ManualLabel']
Target_raw = df['Malfunction']

In [ ]:
# Impute missing (with median)
imp = SimpleImputer(strategy="median")
Features = pd.DataFrame(imp.fit_transform(Features), columns=Features.columns, index=Features.index)
#Features = Features.dropna(axis=1)

# Drop columns that are all NaN or constant
Features = Features.loc[:, Features.notna().any()]  # drop all-NaN
const_mask = Features.nunique(dropna=True) <= 1
if const_mask.any():
    Features = Features.loc[:, ~const_mask]

In [ ]:
Features['Total_timing_delay'] = Features['Brake_timing_delay_exp'] + Features['Release_timing_delay_exp']
Features['Total_energy_delay'] = Features['Brake_energy_delay_exp'] + Features['Release_energy_delay_exp']
Features['Total_power_delay'] = Features['Brake_power_delay_exp'] + Features['Release_power_delay_exp']
Features['Total_power_efficiency'] = Features['Brake_power_efficiency_exp'] + Features['Release_power_efficiency_exp']
Features['Total_energy_efficiency'] = Features['Brake_energy_effiency_exp'] + Features['Release_energy_efficiency_exp']
Features = Features.drop(columns=['Brake_power_efficiency_exp','Release_power_efficiency_exp','Brake_energy_effiency_exp','Release_energy_efficiency_exp','Brake_timing_delay_exp','Release_timing_delay_exp','Brake_energy_delay_exp','Release_energy_delay_exp','Brake_power_delay_exp','Release_power_delay_exp'])

In [ ]:
# === Unified Feature Selection Pipeline: RF, MI, ANOVA
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import mutual_info_classif, f_classif, RFE, RFECV
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt

Xsel = Features.copy()

# Keep only numeric columns (if any non-numeric slipped in)
num_cols = [c for c in Xsel.columns if np.issubdtype(Xsel[c].dtype, np.number)]
Xsel = Xsel[num_cols].copy()

# Some selectors need scaling
sc_std = StandardScaler()
sc_rob = RobustScaler()
# X_std = pd.DataFrame(sc_std.fit_transform(Xsel), columns=Xsel.columns, index=Xsel.index)
# X_rob = pd.DataFrame(sc_rob.fit_transform(Xsel), columns=Xsel.columns, index=Xsel.index)

# if y is string, change to 0/1
y_enc = pd.Series(Target).astype("category")
if y_enc.dtype.name == "category":
    y_enc = y_enc.cat.codes  # e.g., Leakage=1, Normal=0

# For stability on tiny datasets
cv = StratifiedKFold(n_splits=min(5, max(2, np.bincount(y_enc).min())), shuffle=True, random_state=42)

# Helper to convert scores to ranks (lower rank = better)
def to_rank(series, higher_is_better=True):
    s = series.copy()
    if not higher_is_better:
        s = -s
    # rank 1 = best
    return s.rank(ascending=False, method="average")

# ---------- 1) RandomForest importance ----------
# Measures a feature's utility in improving the model's prediction accuracy (e.g., mean decrease in impurity).
# Captures feature interactions naturally; highly effective.
rf = RandomForestClassifier(n_estimators=500, random_state=66, class_weight="balanced")
rf.fit(Xsel, y_enc)
rf_imp = pd.Series(rf.feature_importances_, index=Xsel.columns, name="RF_Importance")
rf_rank = to_rank(rf_imp, higher_is_better=True).rename("RF_Rank")

# ---------- 2) Mutual Information ----------
# Measures statistical dependency (information gain) between a feature and the target.
# Captures non-linear relationships. Evaluates each feature independently; ignores feature interactions.
mi = mutual_info_classif(Xsel, y_enc, random_state=42, discrete_features=False)
mi_score = pd.Series(mi, index=Xsel.columns, name="MI_Score")
mi_rank = to_rank(mi_score, higher_is_better=True).rename("MI_Rank")

# ---------- 3) ANOVA F-test ----------
# (works best if roughly Gaussian/scaled; we used imputed data)
# Measures linear correlation between a feature and the target by comparing variance between groups to variance within groups.
# Assumes linear relationship and Gaussian distribution; ignores feature interactions.
F_vals, p_vals = f_classif(Xsel, y_enc)
f_score = pd.Series(F_vals, index=Xsel.columns, name="ANOVA_F")
f_rank = to_rank(f_score, higher_is_better=True).rename("ANOVA_Rank")

# ---------- Combine all rankings ----------
rank_table = pd.concat([rf_rank, mi_rank, f_rank,
                        rf_imp, mi_score, f_score], axis=1)

# OverallRank: average of available ranks (lower = better)
rank_cols = ["RF_Rank","MI_Rank","ANOVA_Rank"]
rank_table["OverallRank"] = rank_table[rank_cols].mean(axis=1)

# Sort and display top-N
N = 5
rank_table_sorted = rank_table.sort_values("OverallRank").head(N)
print("=== Top features by OverallRank (lower = better) ===")
display(rank_table_sorted)

plt.figure(figsize=(8, max(4, 0.35*N)))
rank_table_sorted.sort_values("OverallRank")["OverallRank"].plot(kind="barh")
plt.gca().invert_yaxis()
plt.title(f"Top {N} Features by Rank")
plt.xlabel("Rank (lower is better)")
plt.tight_layout()
plt.show()

topN_features = rank_table_sorted.index.tolist()
print("\nTopN feature list:", topN_features)

Features_reduced = Features[topN_features]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

Features_reduced = Features_reduced.rename(
    columns={"First_phase_mean_curvature": "FPMC"}
)

sns.heatmap(Features_reduced.corr(), annot=True, cmap='coolwarm')
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.heatmap(Features_reduced.corr(), annot=True, cmap='coolwarm')
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
plt.rcParams.update({
    'axes.titlesize': 22,
    'axes.labelsize': 22,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'axes.facecolor': 'white',
    'figure.facecolor': 'white'
})

In [ ]:
def box_target_plotter(data, target):
    for col in data.select_dtypes("number"):
        sns.boxplot(data=data, x=target, y=col)
        plt.xlabel("")
        plt.show()
        

box_target_plotter(Features_reduced, Target)

In [ ]:
def box_target_plotter(data, target, target_order=None):
    for col in data.select_dtypes("number"):
        sns.boxplot(data=data, x=target, y=col, order=target_order)
        plt.show()

desired_order = ['0', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
box_target_plotter(Features_reduced, Target_raw, target_order=desired_order)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import f_classif
from itertools import product
import numpy as np

levels_weight = Parameters['Weight'].unique()
levels_brake_action = Parameters['Brake_action'].unique()
combinations = list(product(levels_weight, levels_brake_action))

Study_parameter = 'Brake_mode' 
Target_name = Target.name if Target.name else 'Target_Y'
Custom_hue = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

results_column = [
    'Weight', 'Action', 'Top_Feature', 'Max_F_Score',
    'Separability_no_mode',
    'Max_separability_mode_freight', 'Max_separability_mode_passenger', 'Min_separability',
    'Influence_mode_freight', 'Influence_mode_passenger' 
]
df_results = pd.DataFrame(columns=results_column)

for Weight, Action in combinations:
    
    condition_mask = (Parameters['Brake_action'] == Action) & \
                     (Parameters['Weight'] == Weight)
    
    Parameters_controlled = Parameters.loc[condition_mask].copy()
    Features_controlled = Features_reduced.loc[condition_mask].copy()
    Target_controlled = Target.loc[condition_mask].copy()

    if Parameters_controlled.empty:
        continue

    Features_controlled[Study_parameter] = Parameters_controlled[Study_parameter]
    Features_controlled[Target_name] = Target_controlled.values 
    feature_columns = Features_controlled.columns.drop([Study_parameter, Target_name])

    Y_analisi = Features_controlled[Target_name]
    X_analisi = Features_controlled[feature_columns]
    
    if len(Y_analisi.unique()) < 2:
        continue 
    
    # F-score for top feature
    F_vals, p_vals = f_classif(X_analisi, Y_analisi)
    scores = pd.Series(F_vals, index=X_analisi.columns)
    top_scores = scores.sort_values(ascending=False)
    
    max_f_score = top_scores.iloc[0]
    top_feature_name = top_scores.index[0]
    
    # Influence indicator
    mediane_target_senza = Features_controlled.groupby(Target_name)[top_feature_name].median()
    target_levels = mediane_target_senza.index.tolist()
    
    if len(target_levels) != 2: continue # If the Target are not 2 classes
        
    Target_1 = target_levels[0]
    Target_2 = target_levels[1]
    
    # Separability without brake mode
    sep_senza_mode = abs(mediane_target_senza.loc[Target_2] - mediane_target_senza.loc[Target_1])
    
    # Separability with brake mode
    mediane_nidificate = Features_controlled.groupby([Target_name, Study_parameter])[top_feature_name].median()

    try:
        mediana_T1_M0 = mediane_nidificate.loc[(Target_1, 0)]
        mediana_T2_M0 = mediane_nidificate.loc[(Target_2, 0)]
        sep_con_mode_0 = abs(mediana_T2_M0 - mediana_T1_M0)
    except KeyError:
        mediana_T1_M0 = np.nan
        mediana_T2_M0 = np.nan
        sep_con_mode_0 = np.nan
    
    try:
        mediana_T1_M1 = mediane_nidificate.loc[(Target_1, 1)]
        mediana_T2_M1 = mediane_nidificate.loc[(Target_2, 1)]
        sep_con_mode_1 = abs(mediana_T2_M1 - mediana_T1_M1)
    except KeyError:
        mediana_T1_M1 = np.nan
        mediana_T2_M1 = np.nan 
        sep_con_mode_1 = np.nan
    
    if not np.isnan(mediana_T1_M0) and not np.isnan(mediana_T2_M1) and mediana_T1_M0 != 0 and mediana_T2_M1 != 0:
        mediana_diff_T1 = abs(mediana_T1_M0 - mediana_T2_M1)
    else:
        mediana_diff_T1 = np.nan
        
    if not np.isnan(mediana_T1_M1) and not np.isnan(mediana_T2_M0) and mediana_T1_M1 != 0 and mediana_T2_M0 != 0:
        mediana_diff_T2 = abs(mediana_T1_M1 - mediana_T2_M0)
    else:
        mediana_diff_T2 = np.nan
        
    if not np.isnan(mediana_diff_T1) and not np.isnan(mediana_diff_T2):
        min_sep_con_mode = min(mediana_diff_T1, mediana_diff_T2)
    else:
        min_sep_con_mode = np.nan
    
    influenza_mode_freight = ((sep_con_mode_0 - sep_senza_mode)/sep_senza_mode)*100
    influenza_mode_passenger = ((sep_con_mode_1 - sep_senza_mode)/sep_senza_mode)*100

    nuova_riga = pd.DataFrame([{
        'Weight': Weight, 'Action': Action,
        'Top_Feature': top_feature_name,
        'Max_F_Score': max_f_score,
        'Separability_no_mode': sep_senza_mode,
        'Max_separability_mode_freight': sep_con_mode_0,
        'Max_separability_mode_passenger': sep_con_mode_1,
        'Min_separability': min_sep_con_mode,
        'Influence_mode_freight': influenza_mode_freight,
        'Influence_mode_passenger': influenza_mode_passenger
    }])
    df_results = pd.concat([df_results, nuova_riga], ignore_index=True)
    
    plt.figure(figsize=(20, 6)) 
    plt.suptitle(f"Sensitivity analysis | W={Weight}, A={Action} | Feature: {top_feature_name}", fontsize=16)

    # F-Scores
    plt.subplot(1, 3, 1) 
    top_scores.head(5).plot(kind='barh', color='#1f77b4')
    plt.gca().invert_yaxis()
    plt.title("1. Top 5 F-Scores")
    plt.xlabel("F-Score Value")

    # Top Feature vs. Target no Mode
    plt.subplot(1, 3, 2)
    sns.boxplot(data=Features_controlled, x=Target_name, y=top_feature_name, color='#ff7f0e')
    plt.title(f"2. {top_feature_name} vs. Target (Sep. {sep_senza_mode:.3f})")
    plt.xlabel(f"Target ({Target_name})")
    plt.ylabel(top_feature_name)
    plt.grid(axis='y', linestyle='--')

    # Top Feature vs. Target with Mode
    plt.subplot(1, 3, 3)
    sns.boxplot(
        data=Features_controlled,
        x=Target_name,
        y=top_feature_name,
        hue=Study_parameter, 
        palette=Custom_hue
    )
    plt.title(f"3. Interation: Target x {Study_parameter}")
    plt.xlabel(f"Target ({Target_name})")
    plt.ylabel(top_feature_name)
    plt.grid(axis='y', linestyle='--')
    plt.legend(title=Study_parameter)
    
    plt.tight_layout(rect=[0, 0.05, 1, 0.9])
    plt.show()

print("\n\n=======================================================")
print("  Results: Brake mode influence on separability ")
print("=======================================================")
print(df_results)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import f_classif
from itertools import product
import numpy as np

levels_weight = Parameters['Weight'].unique()
levels_brake_action = Parameters['Brake_mode'].unique()
combinations = list(product(levels_weight, levels_brake_action))

Study_parameter = 'Brake_action' 
Target_name = Target.name if Target.name else 'Target_Y'
Custom_hue = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

results_column = [
    'Weight', 'Mode', 'Top_Feature', 'Max_F_Score',
    'Separability_no_mode',
    'Max_separability_action_service', 'Max_separability_action_emergency', 'Min_separability',
    'Influence_action_service', 'Influence_action_emergency' 
]
df_results = pd.DataFrame(columns=results_column)

for Weight, Action in combinations:
    
    condition_mask = (Parameters['Brake_mode'] == Action) & \
                     (Parameters['Weight'] == Weight)
    
    Parameters_controlled = Parameters.loc[condition_mask].copy()
    Features_controlled = Features_reduced.loc[condition_mask].copy()
    Target_controlled = Target.loc[condition_mask].copy()

    if Parameters_controlled.empty:
        continue

    Features_controlled[Study_parameter] = Parameters_controlled[Study_parameter]
    Features_controlled[Target_name] = Target_controlled.values 
    feature_columns = Features_controlled.columns.drop([Study_parameter, Target_name])

    Y_analisi = Features_controlled[Target_name]
    X_analisi = Features_controlled[feature_columns]
    
    if len(Y_analisi.unique()) < 2:
        continue 
    
    # F-score for top feature
    F_vals, p_vals = f_classif(X_analisi, Y_analisi)
    scores = pd.Series(F_vals, index=X_analisi.columns)
    top_scores = scores.sort_values(ascending=False)
    
    max_f_score = top_scores.iloc[0]
    top_feature_name = top_scores.index[0]
    
    # Influence indicator
    mediane_target_senza = Features_controlled.groupby(Target_name)[top_feature_name].median()
    target_levels = mediane_target_senza.index.tolist()
    
    if len(target_levels) != 2: continue # If the Target are not 2 classes
        
    Target_1 = target_levels[0]
    Target_2 = target_levels[1]
    
    # Separability without brake mode
    sep_senza_mode = abs(mediane_target_senza.loc[Target_2] - mediane_target_senza.loc[Target_1])
    
    # Separability with brake mode
    mediane_nidificate = Features_controlled.groupby([Target_name, Study_parameter])[top_feature_name].median()

    try:
        mediana_T1_M0 = mediane_nidificate.loc[(Target_1, 0)]
        mediana_T2_M0 = mediane_nidificate.loc[(Target_2, 0)]
        sep_con_mode_0 = abs(mediana_T2_M0 - mediana_T1_M0)
    except KeyError:
        mediana_T1_M0 = np.nan
        mediana_T2_M0 = np.nan
        sep_con_mode_0 = np.nan
    
    try:
        mediana_T1_M1 = mediane_nidificate.loc[(Target_1, 1)]
        mediana_T2_M1 = mediane_nidificate.loc[(Target_2, 1)]
        sep_con_mode_1 = abs(mediana_T2_M1 - mediana_T1_M1)
    except KeyError:
        mediana_T1_M1 = np.nan
        mediana_T2_M1 = np.nan 
        sep_con_mode_1 = np.nan
    
    if not np.isnan(mediana_T1_M0) and not np.isnan(mediana_T2_M1) and mediana_T1_M0 != 0 and mediana_T2_M1 != 0:
        mediana_diff_T1 = abs(mediana_T1_M0 - mediana_T2_M1)
    else:
        mediana_diff_T1 = np.nan
        
    if not np.isnan(mediana_T1_M1) and not np.isnan(mediana_T2_M0) and mediana_T1_M1 != 0 and mediana_T2_M0 != 0:
        mediana_diff_T2 = abs(mediana_T1_M1 - mediana_T2_M0)
    else:
        mediana_diff_T2 = np.nan
        
    if not np.isnan(mediana_diff_T1) and not np.isnan(mediana_diff_T2):
        min_sep_con_mode = min(mediana_diff_T1, mediana_diff_T2)
    else:
        min_sep_con_mode = np.nan
    
    influenza_mode_freight = ((sep_con_mode_0 - sep_senza_mode)/sep_senza_mode)*100
    influenza_mode_passenger = ((sep_con_mode_1 - sep_senza_mode)/sep_senza_mode)*100

    nuova_riga = pd.DataFrame([{
        'Weight': Weight, 'Mode': Action,
        'Top_Feature': top_feature_name,
        'Max_F_Score': max_f_score,
        'Separability_no_mode': sep_senza_mode,
        'Max_separability_action_service': sep_con_mode_0,
        'Max_separability_action_emergency': sep_con_mode_1,
        'Min_separability': min_sep_con_mode,
        'Influence_action_service': influenza_mode_freight,
        'Influence_action_emergency': influenza_mode_passenger
    }])
    df_results = pd.concat([df_results, nuova_riga], ignore_index=True)
    
    plt.figure(figsize=(20, 6)) 
    plt.suptitle(f"Sensitivity analysis | W={Weight}, M={Action} | Feature: {top_feature_name}", fontsize=16)

    # F-Scores
    plt.subplot(1, 3, 1) 
    top_scores.head(5).plot(kind='barh', color='#1f77b4')
    plt.gca().invert_yaxis()
    plt.title("1. Top 5 F-Scores")
    plt.xlabel("F-Score Value")

    # Top Feature vs. Target no Mode
    plt.subplot(1, 3, 2)
    sns.boxplot(data=Features_controlled, x=Target_name, y=top_feature_name, color='#ff7f0e')
    plt.title(f"2. {top_feature_name} vs. Target (Sep. {sep_senza_mode:.3f})")
    plt.xlabel(f"Target ({Target_name})")
    plt.ylabel(top_feature_name)
    plt.grid(axis='y', linestyle='--')

    # Top Feature vs. Target with Mode
    plt.subplot(1, 3, 3)
    sns.boxplot(
        data=Features_controlled,
        x=Target_name,
        y=top_feature_name,
        hue=Study_parameter, 
        palette=Custom_hue
    )
    plt.title(f"3. Interation: Target x {Study_parameter}")
    plt.xlabel(f"Target ({Target_name})")
    plt.ylabel(top_feature_name)
    plt.grid(axis='y', linestyle='--')
    plt.legend(title=Study_parameter)
    
    plt.tight_layout(rect=[0, 0.05, 1, 0.9])
    plt.show()

print("\n\n=======================================================")
print("  Results: Brake action influence on separability ")
print("=======================================================")
print(df_results)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import f_classif
from itertools import product
import numpy as np

levels_weight = Parameters['Brake_action'].unique()
levels_brake_action = Parameters['Brake_mode'].unique()
combinations = list(product(levels_weight, levels_brake_action))

Study_parameter = 'Weight' 
Target_name = Target.name if Target.name else 'Target_Y'
Custom_hue = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

results_column = [
    'Action', 'Mode', 'Top_Feature', 'Max_F_Score',
    'Separability_no_mode',
    'Max_separability_weight_1', 'Max_separability_weight_2', 'Min_separability',
    'Influence_weight_1', 'Influence_weight_2' 
]
df_results = pd.DataFrame(columns=results_column)

for Weight, Action in combinations:
    
    condition_mask = (Parameters['Brake_mode'] == Action) & \
                     (Parameters['Brake_action'] == Weight)
    
    Parameters_controlled = Parameters.loc[condition_mask].copy()
    Features_controlled = Features_reduced.loc[condition_mask].copy()
    Target_controlled = Target.loc[condition_mask].copy()

    if Parameters_controlled.empty:
        continue

    Features_controlled[Study_parameter] = Parameters_controlled[Study_parameter]
    Features_controlled[Target_name] = Target_controlled.values 
    feature_columns = Features_controlled.columns.drop([Study_parameter, Target_name])

    Y_analisi = Features_controlled[Target_name]
    X_analisi = Features_controlled[feature_columns]
    
    if len(Y_analisi.unique()) < 2:
        continue 
    
    # F-score for top feature
    F_vals, p_vals = f_classif(X_analisi, Y_analisi)
    scores = pd.Series(F_vals, index=X_analisi.columns)
    top_scores = scores.sort_values(ascending=False)
    
    max_f_score = top_scores.iloc[0]
    top_feature_name = top_scores.index[0]
    
    # Influence indicator
    mediane_target_senza = Features_controlled.groupby(Target_name)[top_feature_name].median()
    target_levels = mediane_target_senza.index.tolist()
    
    if len(target_levels) != 2: continue # If the Target are not 2 classes
        
    Target_1 = target_levels[0]
    Target_2 = target_levels[1]
    
    # Separability without brake mode
    sep_senza_mode = abs(mediane_target_senza.loc[Target_2] - mediane_target_senza.loc[Target_1])
    
    # Separability with brake mode
    mediane_nidificate = Features_controlled.groupby([Target_name, Study_parameter])[top_feature_name].median()

    try:
        mediana_T1_M0 = mediane_nidificate.loc[(Target_1, 2.33)]
        mediana_T2_M0 = mediane_nidificate.loc[(Target_2, 2.33)]
        sep_con_mode_0 = abs(mediana_T2_M0 - mediana_T1_M0)
    except KeyError:
        mediana_T1_M0 = np.nan
        mediana_T2_M0 = np.nan
        sep_con_mode_0 = np.nan
    
    try:
        mediana_T1_M1 = mediane_nidificate.loc[(Target_1, 2.5686)]
        mediana_T2_M1 = mediane_nidificate.loc[(Target_2, 2.5686)]
        sep_con_mode_1 = abs(mediana_T2_M1 - mediana_T1_M1)
    except KeyError:
        mediana_T1_M1 = np.nan
        mediana_T2_M1 = np.nan 
        sep_con_mode_1 = np.nan
    
    if not np.isnan(mediana_T1_M0) and not np.isnan(mediana_T2_M1) and mediana_T1_M0 != 0 and mediana_T2_M1 != 0:
        mediana_diff_T1 = abs(mediana_T1_M0 - mediana_T2_M1)
    else:
        mediana_diff_T1 = np.nan
        
    if not np.isnan(mediana_T1_M1) and not np.isnan(mediana_T2_M0) and mediana_T1_M1 != 0 and mediana_T2_M0 != 0:
        mediana_diff_T2 = abs(mediana_T1_M1 - mediana_T2_M0)
    else:
        mediana_diff_T2 = np.nan
        
    if not np.isnan(mediana_diff_T1) and not np.isnan(mediana_diff_T2):
        min_sep_con_mode = min(mediana_diff_T1, mediana_diff_T2)
    else:
        min_sep_con_mode = np.nan
    
    influenza_mode_freight = ((sep_con_mode_0 - sep_senza_mode)/sep_senza_mode)*100
    influenza_mode_passenger = ((sep_con_mode_1 - sep_senza_mode)/sep_senza_mode)*100

    nuova_riga = pd.DataFrame([{
        'Action': Weight, 'Mode': Action,
        'Top_Feature': top_feature_name,
        'Max_F_Score': max_f_score,
        'Separability_no_mode': sep_senza_mode,
        'Max_separability_weight_1': sep_con_mode_0,
        'Max_separability_weight_2': sep_con_mode_1,
        'Min_separability': min_sep_con_mode,
        'Influence_weight_1': influenza_mode_freight,
        'Influence_weight_2': influenza_mode_passenger
    }])
    df_results = pd.concat([df_results, nuova_riga], ignore_index=True)
    
    plt.figure(figsize=(20, 6)) 
    plt.suptitle(f"Sensitivity analysis | A={Weight}, M={Action} | Feature: {top_feature_name}", fontsize=16)

    # F-Scores
    plt.subplot(1, 3, 1) 
    top_scores.head(5).plot(kind='barh', color='#1f77b4')
    plt.gca().invert_yaxis()
    plt.title("1. Top 5 F-Scores")
    plt.xlabel("F-Score Value")

    # Top Feature vs. Target no Mode
    plt.subplot(1, 3, 2)
    sns.boxplot(data=Features_controlled, x=Target_name, y=top_feature_name, color='#ff7f0e')
    plt.title(f"2. {top_feature_name} vs. Target (Sep. {sep_senza_mode:.3f})")
    plt.xlabel(f"Target ({Target_name})")
    plt.ylabel(top_feature_name)
    plt.grid(axis='y', linestyle='--')

    # Top Feature vs. Target with Mode
    plt.subplot(1, 3, 3)
    sns.boxplot(
        data=Features_controlled,
        x=Target_name,
        y=top_feature_name,
        hue=Study_parameter, 
        palette=Custom_hue
    )
    plt.title(f"3. Interation: Target x {Study_parameter}")
    plt.xlabel(f"Target ({Target_name})")
    plt.ylabel(top_feature_name)
    plt.grid(axis='y', linestyle='--')
    plt.legend(title=Study_parameter)
    
    plt.tight_layout(rect=[0, 0.05, 1, 0.9])
    plt.show()

print("\n\n=======================================================")
print("  Results: Weight influence on separability ")
print("=======================================================")
print(df_results)

In [ ]:
level = [2,3]
mal = ['0','A','B','H']
emer = [1]
mode = [0]
df = df_raw[df_raw['Sensor'].isin(level) & 
            df_raw['Malfunction'].isin(mal)]# & 
            #df_raw['Brake_action'].isin(emer) & 
            #df_raw['Brake_mode'].isin(mode)].copy() 

df['ManualLabel'] = np.where(df['Malfunction'].isin(['H']),
                             'Manual brake', 'Healthy')

Features = df.drop(columns=['Malfunction','ManualLabel','Weight','Brake_action','Brake_mode','Frequency','Sensor'])
Parameters = df[['Frequency']].copy() 
Target = df['ManualLabel']
Target_raw = df['Malfunction']

imp = SimpleImputer(strategy="median")
Features = pd.DataFrame(imp.fit_transform(Features), columns=Features.columns, index=Features.index)

Features = Features.loc[:, Features.notna().any()]
const_mask = Features.nunique(dropna=True) <= 1
if const_mask.any():
    Features = Features.loc[:, ~const_mask]

Features_reduced = Features[['First_phase_power','First_phase_half_time_ratio','First_phase_mean_curvature']].copy()

Target_name = 'ManualLabel' if Target.name is None else Target.name

Features_reduced[Target_name] = Target.values 
Features_reduced['Frequency'] = Parameters['Frequency'].values

desired_order = ['Healthy', 'Manual brake'] 

CUSTOM_HUE_PALETTE = ["#1f77b4", "#ff7f0e"]
def box_target_plotter_with_hue(data, target_col_name, hue_column, target_order=None):

    for col in data.columns.drop([target_col_name, hue_column]):
        if data[col].dtype in ['float64', 'int64']:
            plt.figure(figsize=(10, 6)) 
            
            sns.boxplot(data=data, 
                        x=target_col_name, 
                        y=col, 
                        hue=hue_column,
                        order=target_order,
                       palette=CUSTOM_HUE_PALETTE)
            
            plt.title(f'Distribution of {col} per Target, Grouped by {hue_column}')
            plt.xlabel(target_col_name)
            plt.ylabel(col)
            plt.legend(title=hue_column)
            plt.show()

box_target_plotter_with_hue(Features_reduced, Target_name, 'Frequency', target_order=desired_order)

In [ ]:
# Columns we want to summarize
feature_cols = ['First_phase_power',
                'First_phase_half_time_ratio',
                'First_phase_mean_curvature']

# Build median table grouped by Target and Frequency
median_table = (
    Features_reduced
    .groupby([Target_name, 'Frequency'])[feature_cols]
    .median()
    .reset_index()
)

print(median_table)